In [1]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# **Data Processing**

In [4]:
dota2 = pd.read_csv('../data/initial_datasets/dota2/dota2_train_labels_translated_sanitized.csv')

In [10]:
dota2['label'] = dota2['label'].fillna(0)
dota2['label'] = dota2['label'].replace({'x': 1})

/var/folders/lv/pnwq6bmj4tq68bsvy__37qyh0000gn/T/ipykernel_36751/3508702175.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dota2['label'] = dota2['label'].replace({'x': 1})


# **Grouping data**

In [29]:
idx, data = next(iter(dota2.groupby(['match', 'player'])))

In [40]:
data = dota2.groupby(['match', 'player'])['label'].apply(lambda x: int((x == 1).any()))

In [47]:
data['label'].value_counts()

label
0    514
1     71
Name: count, dtype: int64

In [34]:
1 if (data['label'] == 1).any() else 0

0

In [21]:
'. '.join(data['translated_message'].astype(str))

"I'm sure". "that there's a ward here". "Am I right?". "Is there a ward on the high ground?". "When you're up against Pudge, there's always a ward here.". "or is it closer to the rune?". "let's wait". "tinker is muted". lol. "Good luck with your game". "wasn't in mute for no reason". "MID FUCKING NICK". "HAHAHA". "I'm here as the feed."


In [25]:
grouped_texts = dota2.groupby(['match', 'player'])['translated_message'].apply(lambda x: '. '.join(x.astype(str)))
grouped_texts

match  player
0      2         "I'm sure". "that there's a ward here". "Am I ...
       4         "Two-step move SK". "won't work". "how did you...
       6                                             "high ground"
       7         "Place a sentry.". "you'll find out". "Place a...
       9                      "weakest invoker". "like the tinker"
                                       ...                        
99     0                                                     gg wp
       1                                                      ggwp
       2                                                        GG
       6         The message is already in English.. "saw me te...
       7                                                     gg wp
Name: translated_message, Length: 585, dtype: object

In [48]:
grouped_texts = dota2.groupby(['match', 'player'])['translated_message'].apply(lambda x: '. '.join(x.astype(str)))
grouped_labels = dota2.groupby(['match', 'player'])['label'].apply(lambda x: int((x == 1).any()))

In [54]:
grouped_texts = grouped_texts.reset_index()
grouped_labels = grouped_labels.reset_index()



In [55]:
grouped_labels

,match,player,label
0,0,2,0
1,0,4,0
2,0,6,0
3,0,7,0
4,0,9,0
...,...,...,...
580,99,0,0
581,99,1,0
582,99,2,0
583,99,6,0


In [57]:
matches_grouped = pd.merge(grouped_texts, grouped_labels, on=['match', 'player'])

In [58]:
matches_grouped.to_csv('../data/initial_datasets/dota2/dota2_grouped.csv', index=False)

In [65]:
matches_grouped

,match,player,translated_message,label
0,0,2,"""I'm sure"". ""that there's a ward here"". ""Am I ...",0
1,0,4,"""Two-step move SK"". ""won't work"". ""how did you...",0
2,0,6,"""high ground""",0
3,0,7,"""Place a sentry."". ""you'll find out"". ""Place a...",0
4,0,9,"""weakest invoker"". ""like the tinker""",0
...,...,...,...,...
580,99,0,gg wp,0
581,99,1,ggwp,0
582,99,2,GG,0
583,99,6,"The message is already in English.. ""saw me te...",0


In [66]:
matches_grouped['num_words'] = matches_grouped['translated_message'].str.split().str.len()

In [69]:
matches_grouped['num_words'].median()

np.float64(8.0)

In [68]:
print(matches_grouped['num_words'].mean())
print(matches_grouped['num_words'].std())

15.451282051282051
21.402569200630484
